# Model Diagnostics
## Environmental Statistics 6
C.F. Dormann

There are two thing to learn today: 
  * How to simulate data from a fitted model.
  * Using such simulations for model diagnostics.

## 1 Exercise 1: Model diagnostics for the linear model
Use `invented` data (aka simulated or synthetic data), as we then know the truth. For this example, we invent a data set and misspecify the model in order for the model diagnostics to identify a problem.

Specifically, we define two variables, $x_1$ and $x_2$ (both uniform between 0 and 1), and how they affect the response $y$, e.g. $y = 0.7x_1 + 2.5x_2 - 2.5x_2^2 + \varepsilon$ (with $\varepsilon$ = 0.1). To make this fully reproducible, we use a random number seed (`random.seed()`). 

Analyse these data using a correctly specified (G)LM (i.e. with predictors as used for the simulation, but with the coefficients to be estimated from the data).

Instead of the correct model structure, we want to use a misspecified model. Simply omit the quadratic effect, and look at the model output:

The model still looks fine on paper, but now use the four standard model diagnostic plots for the lm to investigate whether it actually is fine (The 4 plots are: Residuals vs. Fitted, Q-Q plot, Scale-location plot, and Standardized residuals vs. leverage plot). As a bonus you can add the plot of Cook's distances versus row labels. Put them into a single figure); do the plots look alright?


Let us still run with this model and see where we get to.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.graphics.regressionplots as pltreg
from scipy.stats import norm, uniform, kstest, probplot
from statsmodels.distributions.empirical_distribution import ECDF

import os
os.environ["R_HOME"] = r"C:\Users\ia1015\.conda\envs\envstats\lib\R"

import warnings


In [3]:
np.random.seed(123)

# define variables






In [4]:
# Analyse data using LM (including all variables)









In [5]:
# Fit a wrong model using LM (excluding X^2 variable)









In [7]:
# carry out the diagnostics: In one plot

# 1. Residuals vs. Fitted 
# 2. Q-Q plot 
# 3. Scale-location plot
# 4. Standardized residuals vs. leverage plot


influence = model.get_influence()
model_frame = influence.summary_frame()
model_frame







## 2 Exercise 2: Simulating data from a GLM
The point now is to invent a new response vector, $y_\text{sim}$, from the fitted model. That is, given our values for $x_1$ and $x_2$, and the fitted coefficients, and the residual noise $\varepsilon$, what *could* the data set have looked like? We’ll go through it in several step.

### 2.1 Simulation by foot
Remember that the analytical computation of the predicted value is simply a matrix multiplication $\hat{y}=Xb$. Note, however, that $X$ contains a column of 1s for the intercept. Simulate $y$ from the fitted model.

This would be one new data set, as could be expected from the model estimates. Compute the correlation with the actual data. Is it well correlated? How well would you expect it to be correlated?

And it is not well correlated. How well *should* it be correlated? Well, on average the correlation should be similar to the R$^2$ of the fitted model: the better the model, the less room there is for simulating the new response.

We can simulate not one but, say, 326 such data sets. I show you two ways to do so, one using a the previous `np.random.normal(0, sd, size=(326, 100)` and the other a `for`-loop. (Note that the simulation ignores the uncertainty of the estimates, and only adds different noise to the simulated data.)


In [8]:
# Simulate responses based on the model assumptions
np.random.seed(234)

X = np.column_stack((np.ones_like(x1), x1, x2)) 

newy = np.dot(X, beta) + np.random.normal(0, sd, 100)






## 3 Exercise 3: Simulating residuals

### 3.1  The probability integral transform with known source distribution
We can convert observations such that they follow a uniform distribution. First, we assume that we know from which distribution the data derive. 
Compute quantiles for the residual, assuming a normal distribution (with mean and sd from the model residuals). Make histogram of the quantiles and add a uniform distribution (U(0,1)) as curve.

### 3.2 The probability integral transform *without* distributional assumption
We now drop the distributional assumption, and use the simulations as forming the distribution from which the data were derived.

Plot the empirical cumulative density distribution for the simulations of the first data point, and indicate the first data points value and its quantile (using `axvline` and `axhline`).

Now, compute this quantile for all data points within their simulations, possibly using a `for`-loop. The plot a histogram of these quantiles.

The final step is to plot these quantiles, which we will start referring to as `simulated residuals`, against the predicted value, to look for patterns. Also, do a formal Kolmogorov-Smirnov-test whether these data deviate from a standard uniform distribution that they are supposed to conform to.


## 4 Exercise 4: DHARMa
The R-package to do this for us is called **DHARMa** and was put together by Florian Hartig (Uni Regensburg). It is extremely useful and adds a few more tests to the simulations and figures presented above. However, there's no equivalent in python, so they plot must be done indivually. Here we present the plots for the model `fmwrong`, where we plot the QQ plot residuals and the Residual vs predicted plot.

### 4.1 DHARMa automatisch
Use DHARMa with the model fmwrong, accepting all its default values. The key functions to use are `simulateResiduals` and `plot` for the resulting object.

We can also switch to R and see what it looks like with the DHARMa package in R

In [90]:
%load_ext rpy2.ipython

C:\Users\ia1015\.conda\envs\envstats2\Lib\site-packages\rpy2\robjects\packages.py:367: UserWarning: The symbol 'quartz' is not in this R namespace/package.
  warnings.warn(


In [91]:
%%R
suppressWarnings(install.packages("DHARMa"))
library(DHARMa)

Exception ignored from cffi callback <function _consolewrite_ex at 0x000001FF7F650720>:
Traceback (most recent call last):
  File "C:\Users\ia1015\.conda\envs\envstats2\Lib\site-packages\rpy2\rinterface_lib\callbacks.py", line 133, in _consolewrite_ex
    s = conversion._cchar_to_str_with_maxlen(buf, n, _CCHAR_ENCODING)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ia1015\.conda\envs\envstats2\Lib\site-packages\rpy2\rinterface_lib\conversion.py", line 138, in _cchar_to_str_with_maxlen
    s = ffi.string(c, maxlen).decode(encoding)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xfc in position 36: invalid start byte
R[write to console]: This is DHARMa 0.4.6. For overview type '?DHARMa'. For recent changes, type news(package = 'DHARMa')



In [9]:
%%R 

#fit and plot the residuals


UsageError: Cell magic `%%R` not found.


### 4.2 DHARMa mit eigenen Simulationen
If the model type is not supported by DHARMa, we can still use its functionality by providing our own simulations,
using createDHARMa. Do this with the simulated data from above.

In [10]:
%%R 

# create DHARMa

UsageError: Cell magic `%%R` not found.


## 5 Exercise 5: DHARMa for discrete data
So far we looked at continuous data. But if we have binary outcomes, the simulations also will only be 0s and 1s, and no quantiles can be computed. So how does DHARMa handle that?

First see that it does, using the `TitanicSurvival` data (from **effects**) and a complex model with interactions.

In [97]:
%%R
install.packages("effects")
library(effects)


R[write to console]: Warning:
R[write to console]:  unable to access index for repository https://mirror.dogado.de/cran/src/contrib:
  cannot open URL 'https://mirror.dogado.de/cran/src/contrib/PACKAGES'

R[write to console]: Warning:
R[write to console]:  unable to access index for repository https://mirror.dogado.de/cran/bin/windows/contrib/4.1:
  cannot open URL 'https://mirror.dogado.de/cran/bin/windows/contrib/4.1/PACKAGES'

Exception ignored from cffi callback <function _consolewrite_ex at 0x000001FF7F650720>:
Traceback (most recent call last):
  File "C:\Users\ia1015\.conda\envs\envstats2\Lib\site-packages\rpy2\rinterface_lib\callbacks.py", line 133, in _consolewrite_ex
    s = conversion._cchar_to_str_with_maxlen(buf, n, _CCHAR_ENCODING)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ia1015\.conda\envs\envstats2\Lib\site-packages\rpy2\rinterface_lib\conversion.py", line 138, in _cchar_to_str_with_maxlen
    s = ffi.string(c, maxlen).decod